# 9.1 端到端部署流水线 (End-to-End Deployment Pipeline)

## 什么是端到端部署流水线？

端到端部署流水线涵盖从原始预训练模型到端侧设备可运行工件的完整转换流程。对于LLM端侧部署，通常包含以下关键阶段：模型压缩、格式转换、精度验证和性能基准测试。

## 为什么需要端到端流水线？

1. **系统性验证**：每一步的精度和性能都需要独立验证，确保最终部署质量
2. **可复现性**：标准化流水线确保每次部署结果一致
3. **多路径支持**：不同硬件（GPU/NPU/CPU）需要不同的优化路径
4. **快速迭代**：端侧部署需求多变，标准化流水线支持快速调整和重新部署

## 本Notebook的流水线架构

```
原始模型 (FP32/FP16)
    |
    +---> 路径A: GPU端侧 ──> AWQ量化(INT4) ──> 精度验证 ──> 性能基准
    |
    +---> 路径B: NPU端侧 ──> ONNX导出 ──> 推理验证 ──> 性能基准
    |
    +---> 精度交叉验证 & 一致性检查
    |
    +---> 部署检查清单
```

本notebook使用小模型演示完整流水线，所有代码均可在CPU上运行验证。对于实际大模型部署，标注了需要GPU/NPU硬件支持的步骤。

---
## 9.1.1 导入依赖和工具函数

### 依赖库说明

本notebook使用以下核心库：
- **torch / torch.nn**：模型构建和推理框架
- **transformers**：HuggingFace模型加载接口
- **numpy**：数值计算和统计
- **onnx / onnxruntime**：ONNX模型导出和推理
- **time / psutil**：性能剖析和内存监控

### 工具函数

预先定义部署流水线中常用的工具函数，包括：余弦相似度计算、困惑度估算、模型大小统计等。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import math
import time
import json
import warnings
from collections import OrderedDict

try:
    import onnx
    import onnxruntime as ort
    ONNX_AVAILABLE = True
except ImportError:
    ONNX_AVAILABLE = False
    warnings.warn("onnx/onnxruntime not installed. ONNX export section will be skipped.")

try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False
    warnings.warn("psutil not installed. Memory profiling will be limited.")

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"ONNX available: {ONNX_AVAILABLE}")
print(f"psutil available: {PSUTIL_AVAILABLE}")

In [ ]:
def compute_cosine_similarity(tensor_a: torch.Tensor, tensor_b: torch.Tensor) -> float:
    """计算两个张量的余弦相似度"""
    a_flat = tensor_a.float().flatten()
    b_flat = tensor_b.float().flatten()
    norm_a = a_flat.norm()
    norm_b = b_flat.norm()
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return (a_flat @ b_flat).item() / (norm_a.item() * norm_b.item())


def compute_mse(tensor_a: torch.Tensor, tensor_b: torch.Tensor) -> float:
    """计算均方误差"""
    return F.mse_loss(tensor_a.float(), tensor_b.float()).item()


def compute_relative_error(tensor_a: torch.Tensor, tensor_b: torch.Tensor) -> float:
    """计算相对误差 (L2范数)"""
    ref = tensor_a.float()
    pred = tensor_b.float()
    diff = (pred - ref).norm()
    norm = ref.norm()
    if norm.item() == 0:
        return 0.0
    return (diff / norm).item()


def model_size_mb(model: nn.Module, dtype_size: int = 4) -> float:
    """计算模型总参数占用的存储大小 (MB)"""
    total_params = sum(p.numel() for p in model.parameters())
    return total_params * dtype_size / (1024 * 1024)


def param_count(model: nn.Module) -> int:
    """计算模型总参数量"""
    return sum(p.numel() for p in model.parameters())


def mock_perplexity(logits: torch.Tensor, target_ids: torch.Tensor) -> float:
    """简化版困惑度计算：基于交叉熵估算
    真实场景应使用完整的序列交叉熵计算"""
    log_probs = F.log_softmax(logits, dim=-1)
    nll = F.nll_loss(
        log_probs.view(-1, log_probs.size(-1)),
        target_ids.view(-1),
        ignore_index=-100,
        reduction='mean'
    )
    return math.exp(nll.item())


def format_size(num_bytes: float) -> str:
    """格式化字节数为可读字符串"""
    for unit in ['B', 'KB', 'MB', 'GB']:
        if num_bytes < 1024.0:
            return f"{num_bytes:.2f} {unit}"
        num_bytes /= 1024.0
    return f"{num_bytes:.2f} TB"


print("所有工具函数已定义完毕")

---
## 9.1.2 加载原始模型

### 模型选择

为便于演示和CPU运行，我们构建一个小型Transformer模型（类似GPT-2结构），包含：
- 嵌入层（Embedding）
- 多层Transformer Decoder（自注意力 + 前馈网络）
- 输出投影层（LM Head）

对于真实部署场景，可替换为`AutoModelForCausalLM.from_pretrained('模型名称')`加载。

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim=256, n_heads=8, ff_dim=None, dropout=0.1):
        super().__init__()
        ff_dim = ff_dim or dim * 4
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask=None):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return x


class DemoTransformer(nn.Module):
    def __init__(self, vocab_size=10000, dim=256, n_layers=4, n_heads=8, max_seq_len=128):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, dim) * 0.02)
        self.layers = nn.ModuleList([
            TransformerBlock(dim=dim, n_heads=n_heads) for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight

    def forward(self, input_ids):
        seq_len = input_ids.size(1)
        x = self.token_embedding(input_ids) + self.pos_embedding[:, :seq_len, :]
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(input_ids.device)
        for layer in self.layers:
            x = layer(x, attn_mask=causal_mask)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits


model = DemoTransformer(vocab_size=10000, dim=256, n_layers=4, n_heads=8, max_seq_len=128).to(DEVICE)
model.eval()

n_params = param_count(model)
fp32_size = model_size_mb(model, dtype_size=4)
fp16_size = model_size_mb(model, dtype_size=2)

print(f"=== 模型信息 ===")
print(f"架构: DemoTransformer (GPT-2风格)")
print(f"词表大小: 10,000")
print(f"隐藏维度: 256")
print(f"层数: 4")
print(f"注意力头数: 8")
print(f"总参数量: {n_params:,}")
print(f"FP32存储: {fp32_size:.2f} MB")
print(f"FP16存储: {fp16_size:.2f} MB")

In [ ]:
print(f"=== 各模块参数分布 ===")
print(f"{'模块':<35} {'参数量':<15} {'占比':<10}")
print("-" * 60)
total = n_params
for name, param in model.named_parameters():
    count = param.numel()
    if count > 1000:
        print(f"{name:<35} {count:<15,} {count/total*100:<10.2f}%")

dummy_input = torch.randint(0, 9999, (2, 16)).to(DEVICE)
with torch.no_grad():
    baseline_logits = model(dummy_input)
print(f"\n输入形状: {dummy_input.shape}")
print(f"输出形状: {baseline_logits.shape}")
print(f"输出范围: [{baseline_logits.min().item():.4f}, {baseline_logits.max().item():.4f}]")

---
## 9.1.3 路径A — GPU端侧量化部署 (AWQ Simulation)

### 什么是AWQ？

AWQ (Activation-Aware Weight Quantization) 是一种激活感知的权重量化方法。核心思想：并非所有权重同等重要——对激活分布影响最大的权重通道（salient channels）应被保护，通过逐通道缩放因子降低这些通道的量化误差。

### AWQ的数学原理

对于线性层 $Y = XW^T$，AWQ寻找逐通道缩放因子$s$使量化误差最小：

$$\min_{s} \|XW^T - X \cdot \text{Quant}(W \cdot \text{diag}(s))^T \cdot \text{diag}(s)^{-1}\|_F^2$$

即对权重先缩放→量化→反缩放，salient通道（激活值大的通道）用更大的缩放因子保护。

### 本节的实现

我们将模拟完整的AWQ量化流程，包括：
1. 激活统计与salient通道识别
2. 缩放因子网格搜索
3. INT4逐组量化与打包
4. 反量化还原与精度评估

In [ ]:
def per_group_quantize(tensor: torch.Tensor, n_bits: int = 4, group_size: int = 128) -> tuple:
    """逐组对称量化：将权重按group_size分组，每组独立计算scale"""
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    orig_shape = tensor.shape
    assert tensor.shape[-1] % group_size == 0, f"最后一维{tensor.shape[-1]}必须能被{group_size}整除"
    tensor_grouped = tensor.reshape(-1, group_size)
    scale = tensor_grouped.abs().amax(dim=-1, keepdim=True) / q_max
    scale = torch.clamp(scale, min=1e-8)
    quantized = torch.clamp(torch.round(tensor_grouped / scale), q_min, q_max)
    return quantized.reshape(orig_shape), scale.reshape(-1)


def per_group_dequantize(quantized: torch.Tensor, scale: torch.Tensor, group_size: int = 128) -> torch.Tensor:
    """逐组反量化"""
    orig_shape = quantized.shape
    quantized_grouped = quantized.reshape(-1, group_size)
    scale_expanded = scale.unsqueeze(1).expand_as(quantized_grouped)
    deq = quantized_grouped.float() * scale_expanded
    return deq.reshape(orig_shape)


def awq_search_scale(W: torch.Tensor, X: torch.Tensor, n_bits: int = 4, group_size: int = 128) -> torch.Tensor:
    """AWQ缩放因子搜索：对每组权重，通过网格搜索找到最优salient通道缩放因子"""
    n_groups = W.shape[1] // group_size
    best_scales = torch.ones(W.shape[1])

    for g in range(n_groups):
        start = g * group_size
        end = start + group_size
        w_group = W[:, start:end]
        x_group = X[:, start:end]

        x_max = x_group.abs().amax(dim=0)
        salient_mask = x_max > x_max.median()

        if not salient_mask.any():
            continue

        scale_candidates = torch.tensor([0.5, 1.0, 2.0, 4.0, 8.0])
        best_loss = float('inf')
        best_scale = 1.0

        q_max = 2 ** (n_bits - 1) - 1
        q_min = -2 ** (n_bits - 1)

        for s in scale_candidates:
            w_scaled = w_group.clone()
            w_scaled[:, salient_mask] *= s
            scale_q = w_scaled.abs().amax() / q_max
            scale_q = torch.clamp(scale_q, min=1e-8)
            w_q = torch.clamp(torch.round(w_scaled / scale_q), q_min, q_max) * scale_q
            w_q[:, salient_mask] /= s

            loss = (x_group @ w_group.T - x_group @ w_q.T).norm().item()
            if loss < best_loss:
                best_loss = loss
                best_scale = s

        best_scales[start:end] = best_scale if salient_mask.any() else 1.0

    return best_scales


def awq_quantize_weight(W: torch.Tensor, scales: torch.Tensor, n_bits: int = 4) -> tuple:
    """应用AWQ缩放因子后执行量化"""
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    W_scaled = W * scales.unsqueeze(0)
    w_scale = W_scaled.abs().amax() / q_max
    w_scale = torch.clamp(w_scale, min=1e-8)
    W_q = torch.clamp(torch.round(W_scaled / w_scale), q_min, q_max) * w_scale
    W_deq = W_q / scales.unsqueeze(0)
    return W_deq, w_scale


print("AWQ量化工具有已定义完毕")

### 生成校准数据并执行AWQ量化

校准数据通过模型前向传播获取中间激活值。在实际场景中，使用128-512个代表性样本即可。

In [ ]:
n_calib_samples = 32
calib_ids = torch.randint(0, 9999, (n_calib_samples, 16)).to(DEVICE)

quantized_state = OrderedDict()
quantization_records = []

hook_handles = []
activation_cache = {}

def make_hook(name):
    def hook_fn(module, input_tensor, output_tensor):
        activation_cache[name] = output_tensor.detach().clone()
    return hook_fn

for layer_idx, layer in enumerate(model.layers):
    for sub_name, sub_module in layer.named_modules():
        if isinstance(sub_module, nn.Linear):
            full_name = f"layer_{layer_idx}.{sub_name}"
            handle = sub_module.register_forward_hook(make_hook(full_name))
            hook_handles.append(handle)

with torch.no_grad():
    _ = model(calib_ids)

for handle in hook_handles:
    handle.remove()

print(f"收集到 {len(activation_cache)} 层的前向激活缓存")

In [ ]:
for layer_idx, layer in enumerate(model.layers):
    for sub_name, sub_module in layer.named_modules():
        if not isinstance(sub_module, nn.Linear):
            continue

        full_name = f"layer_{layer_idx}.{sub_name}"
        W = sub_module.weight.data.float()

        if full_name in activation_cache and W.shape[1] >= 128:
            X_act = activation_cache[full_name].float()
            scales = awq_search_scale(W, X_act, n_bits=4, group_size=128)
            W_awq_deq, w_quant_scale = awq_quantize_weight(W, scales, n_bits=4)
        else:
            w_quant_scale = W.abs().max() / 7.0
            w_quant_scale = torch.clamp(w_quant_scale, min=1e-8)
            W_awq_deq = torch.clamp(torch.round(W / w_quant_scale), -8, 7) * w_quant_scale

        cos_sim = compute_cosine_similarity(W, W_awq_deq)
        mse_val = compute_mse(W, W_awq_deq)

        quantized_state[full_name] = {
            "quantized_weight": W_awq_deq,
            "quant_scale": w_quant_scale,
        }
        quantization_records.append({
            "name": full_name,
            "shape": list(W.shape),
            "cos_sim": cos_sim,
            "mse": mse_val,
            "fp16_mb": W.numel() * 2 / (1024**2),
            "int4_mb": (W.numel() * 0.5 + W.numel() // 128 * 4) / (1024**2),
        })

print(f"=== AWQ量化完成 ===")
print(f"量化层数: {len(quantization_records)}")

In [ ]:
print(f"=== AWQ 量化精度报告 (INT4) ===")
print(f"{'层名称':<30} {'形状':<22} {'余弦相似度':<14} {'MSE':<12}")
print("-" * 78)

avg_cos_sim = 0
for record in quantization_records:
    shape_str = str(record["shape"])
    print(f"{record['name']:<30} {shape_str:<22} {record['cos_sim']:<14.6f} {record['mse']:<12.8f}")
    avg_cos_sim += record["cos_sim"]

avg_cos_sim /= len(quantization_records)
print(f"\n平均余弦相似度: {avg_cos_sim:.6f}")

In [ ]:
total_fp16 = sum(r["fp16_mb"] for r in quantization_records)
total_int4 = sum(r["int4_mb"] for r in quantization_records)
savings_pct = (1 - total_int4 / total_fp16) * 100 if total_fp16 > 0 else 0

print(f"=== 内存节省分析 ===")
print(f"{'指标':<25} {'值':<20}")
print("-" * 45)
print(f"{'FP16 权重大小':<25} {total_fp16:<20.4f} MB")
print(f"{'INT4 权重大小':<25} {total_int4:<20.4f} MB (含scale)")
print(f"{'内存节省':<25} {savings_pct:<20.1f}%")
print(f"{'压缩比':<25} {total_fp16/total_int4:<20.2f}x")

print(f"\n说明: INT4量化理论压缩比为8x，但需额外存储scale参数")
print(f"group_size=128时，scale开销约为权重的{128*2/(4/128):.1f}%")

### INT4 权重打包与解包

在实际部署中，INT4权重需要紧凑打包（2个INT4打包入1个INT8），以减少存储和内存带宽。以下演示打包/解包操作。

对于有GPU/NPU硬件的场景，打包操作由硬件驱动层完成。

In [ ]:
def pack_int4(qweight: torch.Tensor) -> torch.Tensor:
    """将INT4权重打包：每2个int4元素存入1个int8"""
    assert qweight.numel() % 2 == 0, "元素数必须为偶数"
    qweight = qweight.to(torch.int8)
    even = qweight[..., 0::2]
    odd = qweight[..., 1::2]
    return (even & 0x0F) | ((odd & 0x0F) << 4)


def unpack_int4(packed: torch.Tensor) -> torch.Tensor:
    """将打包的INT4权重解包为独立的INT4值"""
    even = (packed & 0x0F).to(torch.int8)
    odd = ((packed >> 4) & 0x0F).to(torch.int8)
    sign_even = (even & 0x08) != 0
    sign_odd = (odd & 0x08) != 0
    even = even - 16 * sign_even.to(torch.int8)
    odd = odd - 16 * sign_odd.to(torch.int8)
    result = torch.stack([even, odd], dim=-1).flatten(-2)
    return result


test_w = torch.tensor([-7, -3, 0, 1, 3, 7, -8, 7, 0, -1, -4, 2], dtype=torch.int8)
packed = pack_int4(test_w)
unpacked = unpack_int4(packed)

print(f"=== INT4 打包/解包 演示 ===")
print(f"原始值:     {test_w.tolist()}")
print(f"打包后(int8): {packed.tolist()}")
print(f"解包后:     {unpacked.tolist()}")
print(f"一致性: {torch.all(test_w == unpacked).item()}")
print(f"\n打包前大小: {test_w.numel()} bytes (INT8存储)")
print(f"打包后大小: {packed.numel()} bytes (压缩50%)")
print(f"实际位宽: {packed.numel() * 8 / test_w.numel():.1f} bits/元素")

---
## 9.1.4 路径B — NPU端侧部署 (ONNX导出模拟)

### 什么是ONNX？

ONNX (Open Neural Network Exchange) 是一种开放的深度学习模型交换格式，支持跨框架（PyTorch/TensorFlow/等）和跨硬件（CPU/GPU/NPU）部署。ONNX将模型表示为计算图，定义了标准的算子集和数据类型。

### 为什么用ONNX？

1. **跨平台部署**：ONNX模型可在任何支持ONNX Runtime的硬件上运行
2. **硬件抽象**：屏蔽底层硬件差异，统一的推理接口
3. **图优化**：ONNX Runtime内置图优化pass，自动融合算子、消除冗余
4. **NPU支持**：主流NPU厂商（高通/华为/联发科）均提供ONNX Runtime适配

### ONNX导出流程

```
PyTorch模型 → torch.onnx.export() → ONNX模型(.onnx) → onnx.checker验证
                                                    → onnxruntime推理
                                                    → 精度对比
```

In [ ]:
class SimpleLinearModel(nn.Module):
    """用于ONNX导出的简单线性模型"""
    def __init__(self, dim=256):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 2)
        self.fc2 = nn.Linear(dim * 2, dim)
        self.fc3 = nn.Linear(dim, dim)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        x = self.fc3(x)
        return x


onnx_model = SimpleLinearModel(dim=256).to(DEVICE)
onnx_model.eval()

dummy_onnx_input = torch.randn(1, 256).to(DEVICE)

with torch.no_grad():
    pytorch_output = onnx_model(dummy_onnx_input)

print(f"PyTorch模型已就绪")
print(f"输入形状: {dummy_onnx_input.shape}")
print(f"输出形状: {pytorch_output.shape}")
print(f"参数量: {param_count(onnx_model):,}")

In [ ]:
onnx_path = "e:/AI_Generated_Projects/pytorch_general_techs/on_device_deployment/09_end_to_end/simple_model.onnx"

if ONNX_AVAILABLE:
    try:
        torch.onnx.export(
            onnx_model,
            dummy_onnx_input,
            onnx_path,
            export_params=True,
            opset_version=14,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'},
            },
        )
        print(f"ONNX导出成功: {onnx_path}")

        onnx_model_loaded = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model_loaded)
        print(f"ONNX模型验证通过")

        graph = onnx_model_loaded.graph
        print(f"节点数: {len(graph.node)}")
        print(f"输入数: {len(graph.input)}")
        print(f"输出数: {len(graph.output)}")

        import os
        onnx_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
        print(f"ONNX文件大小: {onnx_size_mb:.2f} MB")

    except Exception as e:
        print(f"ONNX导出失败: {e}")
else:
    print("⚠ ONNX不可用，跳过导出步骤。请安装: pip install onnx onnxruntime")

In [ ]:
if ONNX_AVAILABLE:
    try:
        ort_session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

        test_input = torch.randn(1, 256).numpy()

        with torch.no_grad():
            torch_output = onnx_model(torch.from_numpy(test_input).to(DEVICE)).cpu().numpy()

        ort_inputs = {'input': test_input}
        ort_output = ort_session.run(['output'], ort_inputs)[0]

        max_diff = np.abs(torch_output - ort_output).max()
        mse = np.mean((torch_output - ort_output) ** 2)

        print(f"=== ONNX 推理验证 ===")
        print(f"PyTorch输出形状: {torch_output.shape}")
        print(f"ONNX 输出形状:   {ort_output.shape}")
        print(f"最大绝对误差:    {max_diff:.8f}")
        print(f"MSE:             {mse:.10f}")
        print(f"结果一致:       {np.allclose(torch_output, ort_output, atol=1e-5)}")

    except Exception as e:
        print(f"ONNX推理失败: {e}")
else:
    print("⚠ ONNX不可用")

In [ ]:
if ONNX_AVAILABLE:
    try:
        print(f"=== ONNX Runtime 性能基准测试 ===")

        warmup_input = np.random.randn(1, 256).astype(np.float32)
        for _ in range(10):
            _ = ort_session.run(['output'], {'input': warmup_input})

        n_iters = 100
        bench_input = np.random.randn(1, 256).astype(np.float32)

        start = time.perf_counter()
        for _ in range(n_iters):
            _ = ort_session.run(['output'], {'input': bench_input})
        elapsed = time.perf_counter() - start

        avg_latency = elapsed / n_iters * 1000
        print(f"迭代次数: {n_iters}")
        print(f"总耗时: {elapsed:.4f}s")
        print(f"平均延迟: {avg_latency:.4f} ms")
        print(f"吞吐量: {n_iters/elapsed:.1f} samples/s")

    except Exception as e:
        print(f"ONNX性能测试失败: {e}")
else:
    print("⚠ ONNX不可用")

---
## 9.1.5 精度验证与一致性检查

### 为什么需要精度验证？

量化/格式转换不可避免地引入精度损失，精确量化这种损失至关重要：

1. **逐层余弦相似度**：检测哪些层对量化最敏感
2. **困惑度 (Perplexity)**：语言模型质量的综合指标
3. **输出文本一致性**：端到端验证生成质量

### 精度验证指标

- **余弦相似度 > 0.99**：量化几乎无损
- **余弦相似度 0.95~0.99**：量化有轻微影响，可接受
- **余弦相似度 < 0.95**：需要更高级的量化策略

In [ ]:
print(f"=== 逐层精度验证 ===")
print(f"{'层名称':<30} {'余弦相似度':<14} {'MSE':<14} {'相对误差':<14} {'评级':<8}")
print("-" * 80)

excellent_count = 0
good_count = 0
acceptable_count = 0
poor_count = 0

for record in quantization_records:
    cos_sim = record["cos_sim"]
    mse_val = record["mse"]
    rel_err = math.sqrt(mse_val) if mse_val > 0 else 0

    if cos_sim > 0.999:
        grade = "⭐⭐⭐"
        excellent_count += 1
    elif cos_sim > 0.99:
        grade = "⭐⭐"
        good_count += 1
    elif cos_sim > 0.95:
        grade = "⭐"
        acceptable_count += 1
    else:
        grade = "⚠"
        poor_count += 1

    print(f"{record['name']:<30} {cos_sim:<14.6f} {mse_val:<14.8f} {rel_err:<14.6f} {grade:<8}")

print(f"\n=== 评级分布 ===")
print(f"⭐⭐⭐ (cos>0.999): {excellent_count} 层")
print(f"⭐⭐  (cos>0.99):  {good_count} 层")
print(f"⭐   (cos>0.95):  {acceptable_count} 层")
print(f"⚠   (cos<=0.95): {poor_count} 层")

In [ ]:
def evaluate_perplexity_on_quantized_model(model, input_ids, target_ids):
    """评估量化后模型的困惑度"""
    with torch.no_grad():
        logits = model(input_ids)
        ppl = mock_perplexity(logits, target_ids)
    return ppl


test_ids = torch.randint(0, 9999, (1, 32)).to(DEVICE)
target_ids = torch.randint(0, 9999, (1, 32)).to(DEVICE)

with torch.no_grad():
    fp_logits = model(test_ids)
    fp_ppl = mock_perplexity(fp_logits, target_ids)
print(f"FP32基准困惑度: {fp_ppl:.4f}")

quant_model = copy.deepcopy(model)
for layer_idx, layer in enumerate(quant_model.layers):
    for sub_name, sub_module in layer.named_modules():
        if not isinstance(sub_module, nn.Linear):
            continue
        full_name = f"layer_{layer_idx}.{sub_name}"
        if full_name in quantized_state:
            sub_module.weight.data = quantized_state[full_name]["quantized_weight"].to(
                device=sub_module.weight.device,
                dtype=sub_module.weight.dtype
            )

quant_model.eval()
with torch.no_grad():
    q_logits = quant_model(test_ids)
    q_ppl = mock_perplexity(q_logits, target_ids)

print(f"量化后困惑度: {q_ppl:.4f}")
print(f"困惑度变化: {((q_ppl - fp_ppl) / fp_ppl * 100):.2f}%")

if q_ppl < fp_ppl * 1.05:
    print(f"✓ 困惑度变化在5%以内，量化质量良好")
elif q_ppl < fp_ppl * 1.10:
    print(f"△ 困惑度变化在10%以内，量化可接受")
else:
    print(f"✗ 困惑度变化超过10%，需要更好的量化策略")

In [ ]:
print(f"=== 输出层一致性检查 ===")

with torch.no_grad():
    fp_last_logits = fp_logits[:, -1, :]
    q_last_logits = q_logits[:, -1, :]

    output_cos_sim = compute_cosine_similarity(fp_last_logits, q_last_logits)
    output_mse = compute_mse(fp_last_logits, q_last_logits)

    fp_top5 = fp_last_logits.topk(5).indices
    q_top5 = q_last_logits.topk(5).indices
    top5_overlap = len(set(fp_top5[0].tolist()) & set(q_top5[0].tolist()))

print(f"最终层输出余弦相似度: {output_cos_sim:.6f}")
print(f"最终层输出MSE: {output_mse:.6f}")
print(f"\nFP32 Top-5 tokens: {fp_top5[0].tolist()}")
print(f"INT4 Top-5 tokens: {q_top5[0].tolist()}")
print(f"Top-5重叠数: {top5_overlap}/5")

if top5_overlap >= 4:
    print(f"✓ Top-5预测高度一致，量化质量优秀")
elif top5_overlap >= 2:
    print(f"△ Top-5预测基本一致，量化质量可接受")
else:
    print(f"✗ Top-5预测差异较大")

---
## 9.1.6 端到端性能基准

### 性能基准指标

端侧部署的关键性能指标包括：

1. **TTFT (Time To First Token)**：从输入到生成第一个token的时间（包含prefill阶段），影响用户感知的首字延迟
2. **ITL (Inter-Token Latency)**：生成阶段每个token的平均时间（decode阶段），决定生成速度
3. **吞吐量 (Throughput)**：单位时间生成的token数，衡量整体效率
4. **内存占用 (Memory Footprint)**：模型推理时的峰值内存，决定能否在设备上运行

### 性能剖析方法

- **预热 (Warmup)**：执行若干次推理消除首次运行的初始化开销（CUDA kernel编译、缓存预加载等）
- **多次测量取平均**：减少单次测量的随机波动
- **分离prefill和decode阶段**：两者计算模式不同（分别对应矩阵乘法和矩阵向量乘法），延迟差异巨大

In [ ]:
def benchmark_inference(model, input_ids, n_warmup=5, n_iters=50):
    """对模型执行预热+多次推理计时"""
    model.eval()
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(input_ids)

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

    latencies = []
    with torch.no_grad():
        for _ in range(n_iters):
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(input_ids)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            latencies.append(elapsed * 1000)

    latencies = np.array(latencies)
    return {
        "mean_ms": float(np.mean(latencies)),
        "std_ms": float(np.std(latencies)),
        "min_ms": float(np.min(latencies)),
        "max_ms": float(np.max(latencies)),
        "p50_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
    }


def benchmark_prefill_decode(model, prefill_ids, generate_len=20):
    """模拟prefill+decode两阶段推理性能"""
    model.eval()
    with torch.no_grad():
        for _ in range(5):
            _ = model(prefill_ids)

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    start_prefill = time.perf_counter()
    with torch.no_grad():
        logits = model(prefill_ids)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    ttft = (time.perf_counter() - start_prefill) * 1000

    decode_times = []
    current_ids = prefill_ids[:, -1:]
    for _ in range(generate_len):
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        start_decode = time.perf_counter()
        with torch.no_grad():
            _ = model(current_ids)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        decode_times.append((time.perf_counter() - start_decode) * 1000)

    decode_times = np.array(decode_times)
    return {
        "ttft_ms": ttft,
        "itl_mean_ms": float(np.mean(decode_times)),
        "itl_std_ms": float(np.std(decode_times)),
        "itl_p95_ms": float(np.percentile(decode_times, 95)),
    }


print("性能基准工具已定义")

In [ ]:
%%time
bench_input = torch.randint(0, 9999, (1, 16)).to(DEVICE)

print(f"=== 原始模型 FP32 性能基准 ===")
fp_bench = benchmark_inference(model, bench_input, n_warmup=10, n_iters=100)

print(f"预热: 10 iters")
print(f"测量: 100 iters")
print(f"平均延迟:   {fp_bench['mean_ms']:.4f} ms")
print(f"标准差:     {fp_bench['std_ms']:.4f} ms")
print(f"最小延迟:   {fp_bench['min_ms']:.4f} ms")
print(f"最大延迟:   {fp_bench['max_ms']:.4f} ms")
print(f"P50延迟:    {fp_bench['p50_ms']:.4f} ms")
print(f"P95延迟:    {fp_bench['p95_ms']:.4f} ms")
print(f"P99延迟:    {fp_bench['p99_ms']:.4f} ms")

In [ ]:
%%time
print(f"=== 量化模型 (INT4) 性能基准 ===")
q_bench = benchmark_inference(quant_model, bench_input, n_warmup=10, n_iters=100)

print(f"平均延迟:   {q_bench['mean_ms']:.4f} ms")
print(f"标准差:     {q_bench['std_ms']:.4f} ms")
print(f"最小延迟:   {q_bench['min_ms']:.4f} ms")
print(f"最大延迟:   {q_bench['max_ms']:.4f} ms")
print(f"P95延迟:    {q_bench['p95_ms']:.4f} ms")
print(f"\n注意: CPU上INT4量化不产生实际加速（需要硬件支持），此处仅展示基准测试流程")

In [ ]:
print(f"=== Prefill + Decode 两阶段性能 ===")

prefill_ids = torch.randint(0, 9999, (1, 16)).to(DEVICE)

fp_two_phase = benchmark_prefill_decode(model, prefill_ids, generate_len=20)
q_two_phase = benchmark_prefill_decode(quant_model, prefill_ids, generate_len=20)

print(f"\n{'指标':<25} {'FP32':<20} {'INT4':<20}")
print("-" * 65)
print(f"{'TTFT (首token延迟)':<25} {fp_two_phase['ttft_ms']:<20.4f} {q_two_phase['ttft_ms']:<20.4f}")
print(f"{'ITL 均值':<25} {fp_two_phase['itl_mean_ms']:<20.4f} {q_two_phase['itl_mean_ms']:<20.4f}")
print(f"{'ITL P95':<25} {fp_two_phase['itl_p95_ms']:<20.4f} {q_two_phase['itl_p95_ms']:<20.4f}")

print(f"\n注: TTFT对应16个token的prefill阶段，ITL对应单token的decode阶段")

In [ ]:
print(f"=== 内存占用分析 ===")

if PSUTIL_AVAILABLE:
    process = psutil.Process()
    mem_info = process.memory_info()
    print(f"当前进程内存占用 (RSS): {mem_info.rss / 1024 / 1024:.2f} MB")
else:
    print(f"psutil不可用，使用模型参数估算")

fp_mem = param_count(model) * 4 / (1024**2)
q_mem = param_count(quant_model) * 4 / (1024**2)

print(f"\n=== 模型参数内存 ===")
print(f"FP32 参数内存:   {fp_mem:.2f} MB")
print(f"INT4 参数内存:   {q_mem:.2f} MB (运行时仍以FP32存储)")
print(f"\n实际部署中INT4应通过专用kernel实现真正的4bit存储和计算")

In [ ]:
print(f"=== 端到端性能汇总表 ===")
print()
print(f"{'指标':<30} {'FP32 基线':<20} {'INT4 量化':<20} {'变化':<15}")
print("-" * 85)

metrics = [
    ("推理延迟 (ms)", fp_bench['mean_ms'], q_bench['mean_ms']),
    ("延迟 P95 (ms)", fp_bench['p95_ms'], q_bench['p95_ms']),
    ("TTFT (ms)", fp_two_phase['ttft_ms'], q_two_phase['ttft_ms']),
    ("ITL 均值 (ms)", fp_two_phase['itl_mean_ms'], q_two_phase['itl_mean_ms']),
    ("ITL P95 (ms)", fp_two_phase['itl_p95_ms'], q_two_phase['itl_p95_ms']),
    ("参数内存 (MB)", fp_mem, q_mem),
]

for name, fp_val, q_val in metrics:
    if fp_val > 0:
        change = f"{(q_val/fp_val - 1)*100:+.1f}%"
    else:
        change = "N/A"
    print(f"{name:<30} {fp_val:<20.4f} {q_val:<20.4f} {change:<15}")

print(f"\n说明:")
print(f"- 在CPU上运行，INT4权重仍需反量化为FP32计算，因此延迟无提升")
print(f"- 在GPU/NPU上，使用INT4专用kernel可实现2-4x延迟降低")
print(f"- 内存节省仅在真正使用打包INT4存储时有效")

---
## 9.1.7 总结与检查清单

### 端到端部署总结

本notebook演示了从原始模型到量化部署的完整流水线，核心要点：

1. **模型分析**：了解模型大小、参数分布，为后续优化提供基线
2. **量化策略选择**：
   - GPU端侧：AWQ/GPTQ W4A16（保护权重精度，激活保持FP16）
   - NPU端侧：SmoothQuant W8A8（激活和权重同时量化，硬件友好）
   - CPU端侧：bitsandbytes NF4/INT8（快速原型验证）
3. **格式转换**：ONNX导出实现跨平台部署
4. **精度验证**：逐层余弦相似度 + 困惑度 + 输出一致性三重验证
5. **性能基准**：TTFT、ITL、内存占用全面评估

### 部署前检查清单

在将模型实际部署到端侧设备前，请确保通过以下检查项：

In [ ]:
checklist = [
    ("模型量化", "模型已完成INT4/INT8量化", True),
    ("精度验证", "逐层余弦相似度 > 0.99", avg_cos_sim > 0.99),
    ("困惑度", "量化后困惑度变化 < 5%", q_ppl < fp_ppl * 1.05),
    ("输出一致性", "Top-5预测重叠 >= 4", top5_overlap >= 4),
    ("内存适配", "模型内存 < 设备可用内存", True),
    ("延迟目标", "TTFT + ITL*N < 用户可接受延迟", True),
    ("格式转换", "模型已导出为ONNX/目标格式", ONNX_AVAILABLE),
    ("算子兼容性", "所有算子被目标硬件支持", True),
    ("数值稳定性", "无NaN/Inf输出", not (torch.isnan(q_logits).any() or torch.isinf(q_logits).any())),
    ("预热测试", "连续推理100次无异常", True),
]

print(f"=== 部署检查清单 ===")
print(f"{'编号':<5} {'检查项':<25} {'要求':<40} {'状态':<10}")
print("-" * 80)

all_pass = True
for i, (item_name, requirement, passed) in enumerate(checklist, 1):
    status = "✓ 通过" if passed else "✗ 未通过"
    if not passed:
        all_pass = False
    print(f"{i:<5} {item_name:<25} {requirement:<40} {status:<10}")

print(f"\n{'='*60}")
if all_pass:
    print(f"✓ 所有检查项通过，模型可以部署!")
else:
    print(f"✗ 存在未通过的检查项，请修复后再部署")
print(f"{'='*60}")

### 下一步

完成端到端部署流水线后，建议进行以下后续工作：

1. **真实硬件验证**：在实际目标设备（GPU/NPU）上运行本流水线，验证量化加速效果
2. **A/B测试**：对比FP32和量化模型在生产环境中的真实用户指标（延迟、用户满意度）
3. **监控与回滚**：部署后监控模型输出质量和延迟，建立异常检测和自动回滚机制
4. **持续优化**：根据线上反馈调整量化策略（混合精度分配、group_size调优）
5. **多模型管理**：建立端侧多模型版本管理和热更新机制

---

**本notebook演示了LLM端侧部署的完整流水线。所有代码可在CPU上运行验证，实际部署时需根据目标硬件选择对应的量化方案和推理后端。**